[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/qwen3-sft-collab/src/MaxText/examples/sft_qwen3_demo.ipynb)

# Qwen3-0.6B Supervised Fine-Tuning (SFT) Demo


## Overview

This notebook can run on the **public TPU 5e-1**

This notebook demonstrates how to perform Supervised Fine-Tuning (SFT) on Qwen3-0.6B using the Hugging Face ultrachat_200k dataset with Tunix integration for efficient training.

## Dataset Overview

**Dataset Link:** https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k

### Dataset Information:
- **Name:** HuggingFaceH4/ultrachat_200k
- **Type:** Supervised Fine-Tuning dataset
- **Size:** ~200k conversations
- **Format:** Chat conversations with human-AI pairs
- **Splits:** train_sft, test_sft
- **Data columns:** ['messages']

### Dataset Structure:
Each example contains a 'messages' field with:
- **role:** 'user' or 'assistant'
- **content:** The actual message text

### Example data format:
```json
{
  "messages": [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."}
  ]
}
```

## Prerequisites
- HuggingFace access token for dataset download
- Sufficient compute resources (TPU/GPU)

In [ ]:
### (Optional) Run this if you just have this file and nothing else

# 1. Clone the MaxText repository (from AI‑Hypercomputer)
!git clone https://github.com/AI-Hypercomputer/maxtext.git

# 2. Navigate into the cloned directory
%cd maxtext

In [ ]:
### (Optional) Do not run this if you already installed the dependencies

# 3. Ensure setup.sh is executable
!chmod +x setup.sh

# 4. Execute the setup script
!./setup.sh

# force numpy version
!pip install --force-reinstall numpy==2.1.2
#install nest_asyncio
!pip install nest_asyncio

import nest_asyncio
nest_asyncio.apply()
# To fix "This event loop is already running" error in Colab


In [ ]:
#Set up the variables for the script
import os
import sys

#Set the MaxText home directory (where you cloned the maxtext repo)
# once running the jupyter notebook you can use 
# MAXTEXT_REPO_ROOT = os.path.expanduser("~") + "/maxtext"
# This one is for colab
MAXTEXT_REPO_ROOT = os.path.join("/content", "maxtext")

print(f"MaxText Home directory (from Python): {MAXTEXT_REPO_ROOT}")

DEBUG = False  # set to True to run in debug mode, for more print statements
#set this to the path of the checkpoint you want to load, gs:// supported 
#MODEL_CHECKPOINT_PATH = "path/to/scanned/checkpoint" 
# now for colab we will use the checkpoint from the HF model
MODEL_CHECKPOINT_PATH = f"{MAXTEXT_REPO_ROOT}/qwen_checkpoint"

In [ ]:
# Hugging Face Authentication Setup
from huggingface_hub import login

# Set your Hugging Face token here
HF_TOKEN = "your_actual_token_here"  # Replace with your actual token
login(token=HF_TOKEN)

In [ ]:
# This is the command to convert the HF model to the MaxText format 
# You may omit it if you already have a checkpoint
!python3 -m MaxText.utils.ckpt_conversion.to_maxtext \
    $MAXTEXT_REPO_ROOT/src/MaxText/configs/base.yml \
    model_name=qwen3-0.6b \
    base_output_directory=$MODEL_CHECKPOINT_PATH \
    hf_access_token=$HF_TOKEN \
    use_multimodal=false \
    scan_layers=false

In [ ]:
from pathlib import Path
from typing import Optional, Dict, Any

# Find MaxText directory and change working directory to it
current_dir = Path.cwd()

maxtext_path = Path(f'{MAXTEXT_REPO_ROOT}') / 'src' / 'MaxText'

# Change working directory to MaxText project root
os.chdir(maxtext_path)
sys.path.insert(0, str(maxtext_path))

print(f"✓ Changed working directory to: {os.getcwd()}")
print(f"✓ MaxText project root: {maxtext_path}")
print(f"✓ Added to Python path: {maxtext_path}")
import jax
if not jax.distributed.is_initialized():
    jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
# MaxText imports
try:
    from MaxText import pyconfig
    from MaxText.sft.sft_trainer import train as sft_train

    MAXTEXT_AVAILABLE = True
    print("✓ MaxText imports successful")
except ImportError as e:
    print(f"⚠️ MaxText not available: {e}")
    MAXTEXT_AVAILABLE = False

In [ ]:
# Fixed configuration setup for Qwen-0.6B on small TPU
if MAXTEXT_AVAILABLE:
    config_argv = [
        "",
        f"{MAXTEXT_REPO_ROOT}/src/MaxText/configs/sft.yml",   # base SFT config
        f"load_parameters_path={MODEL_CHECKPOINT_PATH}/0/items/",  # Load pre-trained weights!, replace with your checkpoint path
        "model_name=qwen3-0.6b",
        "steps=20",                                     # very short run for testing
        "per_device_batch_size=1",                      # minimal to avoid OOM
        "max_target_length=1024",                        
        "learning_rate=2.0e-5",                         # safe small LR
        "eval_steps=5",
        "weight_dtype=bfloat16",
        "dtype=bfloat16",
        "hf_path=HuggingFaceH4/ultrachat_200k",                       # HuggingFace dataset/model if needed
        f"hf_access_token={HF_TOKEN}",
        "base_output_directory=/tmp/maxtext_qwen06",
        "run_name=sft_qwen0.6b_test",
        "tokenizer_path=Qwen/Qwen3-0.6B",                # Qwen tokenizer
        "eval_interval=10",
        "profiler=xplane",
    ]

    # Initialize configuration using MaxText's pyconfig
    config = pyconfig.initialize(config_argv)

    print("✓ Fixed configuration loaded:")
    print(f"  - Model: {config.model_name}")
    print(f"  - Dataset: {config.hf_path}")
    print(f"  - Steps: {config.steps}")
    print(f"  - Use SFT: {config.use_sft}")
    print(f"  - Learning Rate: {config.learning_rate}")
else:
    print("MaxText not available - cannot load configuration")

In [ ]:
#  Execute the training using MaxText SFT trainer's train() function
if MAXTEXT_AVAILABLE:
    print("="*60)
    print("EXECUTING ACTUAL TRAINING")
    print("="*60)

    sft_train(config)
